This project is about the sentiment140 dataset.
It contains 1,600,000 tweets

**Set up your environment**

In [4]:
#import necessary libraries and load dataset 
import pandas as pd

    
# The original CSV has six columns and no header row.
columns = ["sentiment", "id", "date", "query", "user", "text"]


#load teh dataset
df = pd.read_csv(
    "training.1600000.processed.noemoticon.csv",
    encoding="latin-1",
    header=None,
    names=columns
)

# Keep the target label and tweet text.
df = df[["sentiment", "text"]].copy()

print("Dataset shape:", df.shape)
print("\nSentiment counts:")
print(df["sentiment"].value_counts())

df.head()

Dataset shape: (1600000, 2)

Sentiment counts:
sentiment
0    800000
4    800000
Name: count, dtype: int64


,sentiment,text
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,is upset that he can't update his Facebook by ...
2,0,@Kenichan I dived many times for the ball. Man...
3,0,my whole body feels itchy and like its on fire
4,0,"@nationwideclass no, it's not behaving at all...."


**removes missing values and clean the dataset by also  removing the URl, special characters, empty tweet and duplicated tweets**

In [5]:
import re
import html

# Keep the tweet text and sentiment label.
clean_df = df[["sentiment", "text"]].copy()
original_rows = len(clean_df)

# Remove missing text or sentiment labels.
clean_df = clean_df.dropna(subset=["text", "sentiment"])

# Keep valid Sentiment140 labels.
clean_df = clean_df[clean_df["sentiment"].isin([0, 2, 4])].copy()

def clean_tweet(text):
    # Convert HTML entities: &amp; becomes &.
    text = html.unescape(str(text))

    # Remove URLs and mentions.
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)

    # Convert to lowercase.
    text = text.lower()

    # Preserve negative words before removing punctuation.
    text = text.replace("’", "'")
    text = re.sub(r"\bcan't\b", "can not", text)
    text = re.sub(r"\bwon't\b", "will not", text)
    text = re.sub(r"n't\b", " not", text)

    # Remove special characters; keep letters, numbers, and spaces.
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Remove extra spaces.
    return re.sub(r"\s+", " ", text).strip()

# Clean tweets while keeping the original text.
clean_df["clean_text"] = clean_df["text"].apply(clean_tweet)

# Remove tweets that become empty after cleaning.
clean_df = clean_df[clean_df["clean_text"].ne("")].copy()

# Remove repeated cleaned tweets with the same sentiment.
clean_df = clean_df.drop_duplicates(
    subset=["clean_text", "sentiment"]
).reset_index(drop=True)

# Tokenize: split each cleaned tweet into words.
clean_df["tokens"] = clean_df["clean_text"].str.split()

# Display the results.
print("Original rows:", original_rows)
print("Cleaned rows:", len(clean_df))
print("Removed rows:", original_rows - len(clean_df))

print("\nSentiment counts:")
print(clean_df["sentiment"].value_counts())

clean_df[["sentiment", "text", "clean_text", "tokens"]].head()

Original rows: 1600000
Cleaned rows: 1524442
Removed rows: 75558

Sentiment counts:
sentiment
0    767308
4    757134
Name: count, dtype: int64


,sentiment,text,clean_text,tokens
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",awww that s a bummer you shoulda got david car...,"[awww, that, s, a, bummer, you, shoulda, got, ..."
1,0,is upset that he can't update his Facebook by ...,is upset that he can not update his facebook b...,"[is, upset, that, he, can, not, update, his, f..."
2,0,@Kenichan I dived many times for the ball. Man...,i dived many times for the ball managed to sav...,"[i, dived, many, times, for, the, ball, manage..."
3,0,my whole body feels itchy and like its on fire,my whole body feels itchy and like its on fire,"[my, whole, body, feels, itchy, and, like, its..."
4,0,"@nationwideclass no, it's not behaving at all....",no it s not behaving at all i m mad why am i h...,"[no, it, s, not, behaving, at, all, i, m, mad,..."


Note that tokenization breaks text into smaller pieces called tokens and as instructed, the tokens are converted to lowercase

**Train and validate logistic regression**

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Remove texts with conflicting labels and repeated cleaned texts and prevents identical tweets appearing in both splits.
label_counts = clean_df.groupby("clean_text")["sentiment"].nunique()
conflicting_texts = label_counts[label_counts > 1].index

model_df = (
    clean_df[~clean_df["clean_text"].isin(conflicting_texts)]
    .drop_duplicates(subset="clean_text")
    .copy()
)

# Input: cleaned tweets. Target: original sentiment labels.
X = model_df["clean_text"]
y = model_df["sentiment"]

# Use 80% for training and 20% for validation.
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(  #TF-IDF converts tweets into numerical features based on their words.
        max_features=50000,
        ngram_range=(1, 2),
        min_df=2,
        sublinear_tf=True
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        solver="lbfgs"
    ))
])

# Fit TF-IDF and logistic regression using training data only.
model.fit(X_train, y_train)

# Evaluate on separate validation data.
predictions = model.predict(X_val)

print(f"Validation accuracy: {accuracy_score(y_val, predictions):.4f}")
print(f"Validation Macro F1: {f1_score(y_val, predictions, average='macro'):.4f}")

sentiment_names = {0: "negative", 2: "neutral", 4: "positive"}
labels = model.classes_.tolist()

print(classification_report(
    y_val,
    predictions,
    labels=labels,
    target_names=[sentiment_names[label] for label in labels],
    zero_division=0
))

Validation accuracy: 0.8159
Validation Macro F1: 0.8159
              precision    recall  f1-score   support

    negative       0.82      0.81      0.82    152648
    positive       0.81      0.83      0.82    150613

    accuracy                           0.82    303261
   macro avg       0.82      0.82      0.82    303261
weighted avg       0.82      0.82      0.82    303261



**API Development and Containerization**

In [8]:
#saved the trained pipeline
import joblib

joblib.dump(model, "sentiment_model.joblib")
print("Model saved.")

Model saved.


In [10]:
%%writefile preprocessing.py
import html
import re

def clean_tweet(text):
    text = html.unescape(str(text))
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = text.lower()

    text = text.replace("’", "'")
    text = re.sub(r"\bcan't\b", "can not", text)
    text = re.sub(r"\bwon't\b", "will not", text)
    text = re.sub(r"n't\b", " not", text)

    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

Writing preprocessing.py


**Create the Flask API**

In [18]:
%%writefile app.py
from pathlib import Path

import joblib
from flask import Flask, jsonify, request

from preprocessing import clean_tweet

app = Flask(__name__)

model_path = Path(__file__).resolve().with_name("sentiment_model.joblib")
model = joblib.load(model_path)

sentiment_names = {
    0: "negative",
    2: "neutral",
    4: "positive"
}

@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json(silent=True)

    if not isinstance(data, dict):
        return jsonify({"error": "Send a JSON object."}), 400

    tweet = data.get("tweet")

    if not isinstance(tweet, str) or not tweet.strip():
        return jsonify({"error": "Provide a non-empty 'tweet' string."}), 400

    cleaned = clean_tweet(tweet)

    if not cleaned:
        return jsonify({"error": "Tweet contains no usable text."}), 400

    label = int(model.predict([cleaned])[0])

    return jsonify({
        "tweet": tweet,
        "prediction": label,
        "sentiment": sentiment_names[label]
    })
@app.route("/health", methods=["GET"])
def health():
    return jsonify({
        "status": "ok",
        "model_version": os.environ.get("MODEL_VERSION", "local")
    })
    
if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

Overwriting app.py


In [19]:
#Create  the dependenciies
from importlib.metadata import version
from pathlib import Path

packages = ["Flask", "scikit-learn", "numpy", "scipy", "joblib"]

requirements = [
    f"{package}=={version(package)}"
    for package in packages
]
requirements.append("gunicorn==23.0.0")

Path("requirements.txt").write_text(
    "\n".join(requirements) + "\n",
    encoding="utf-8"
)

print(Path("requirements.txt").read_text())

Flask==3.1.3
scikit-learn==1.9.1
numpy==2.4.6
scipy==1.18.1
joblib==1.6.0
gunicorn==23.0.0



**#create the Dokerfile**

In [20]:
%%writefile Dockerfile
FROM python:3.14-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py preprocessing.py sentiment_model.joblib ./

EXPOSE 5000

CMD ["gunicorn", "--bind", "0.0.0.0:5000", "--workers", "1", "app:app"]

Overwriting Dockerfile


In [21]:
import sys
print(f"{sys.version_info.major}.{sys.version_info.minor}")

3.14


**Create .dockerignore to exclude unnecessary files from the build:**

In [22]:
%%writefile .dockerignore
.git/
.venv/
__pycache__/
.ipynb_checkpoints/
*.ipynb
*.csv
data/

Overwriting .dockerignore


**Create API tests**

In [24]:
#create a folder name test
from pathlib import Path

Path("tests").mkdir(exist_ok=True)

In [25]:
%%writefile tests/test_api.py
import pytest
from app import app

@pytest.fixture
def client():
    app.config["TESTING"] = True
    with app.test_client() as client:
        yield client

def test_health(client):
    response = client.get("/health")

    assert response.status_code == 200
    assert response.get_json()["status"] == "ok"

@pytest.mark.parametrize(
    "tweet, expected_sentiment",
    [
        ("I love this wonderful product", "positive"),
        ("I hate this terrible product", "negative"),
    ]
)
def test_sentiment(client, tweet, expected_sentiment):
    response = client.post("/predict", json={"tweet": tweet})

    assert response.status_code == 200
    result = response.get_json()
    assert result["sentiment"] == expected_sentiment
    assert result["tweet"] == tweet

def test_missing_tweet(client):
    response = client.post("/predict", json={})
    assert response.status_code == 400

def test_empty_tweet(client):
    response = client.post("/predict", json={"tweet": "   "})
    assert response.status_code == 400

def test_invalid_type(client):
    response = client.post("/predict", json={"tweet": 123})
    assert response.status_code == 400

def test_invalid_json(client):
    response = client.post(
        "/predict",
        data="not JSON",
        content_type="application/json"
    )
    assert response.status_code == 400

Writing tests/test_api.py


In [26]:
import sys
import subprocess

subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"])

CompletedProcess(args=['C:\\Users\\tebos\\AppData\\Local\\Python\\pythoncore-3.14-64\\python.exe', '-m', 'pytest', '-q', 'tests'], returncode=1)

In [28]:
import os
from pathlib import Path

import joblib
from flask import Flask, jsonify, request